# Semantic Benchmark Java Tuning Analysis

Reads `04_tune_pss_wasserstein.py` outputs, plots pair score distributions with thresholds when both labels exist, and renders available confusion examples with code, graphs, eigenvalues, and metric scores.

In [ ]:
from pathlib import Path
import html
import json
import pickle
import random
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import HTML, display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "spectral_code").exists() and (candidate / "pipelines").exists():
            return candidate
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from spectral_code.evaluation.notebook_helpers import semantic_spec

LANGUAGE = "java"
DISPLAY_LANGUAGE = "Java"
GRAPH_TYPES = ['ast', 'cfg', 'ddg', 'pdg', 'cpg']
PRIMARY_GRAPH_TYPE = "cpg"

spec = semantic_spec(LANGUAGE)
DATA_DIR = spec.data_dir
OUTPUT_ROOT = spec.output_root
REPORTS_DIR = OUTPUT_ROOT / "reports"
GRAPH_MANIFEST_PATH = OUTPUT_ROOT / "clean_graphs" / "graph_shards_manifest.json"
SPECTRAL_MANIFEST_PATH = OUTPUT_ROOT / "spectral_features" / "spectral_features_manifest.json"
PIPELINE_TIMINGS_PATH = OUTPUT_ROOT / "pipeline_timings.json"

print("Project root:", PROJECT_ROOT)
print("Prepared data:", DATA_DIR)
print("Output root:", OUTPUT_ROOT)


In [ ]:
def load_code_for_ids(path: Path, wanted_ids: set[str]) -> dict[str, str]:
    code_map = {}
    if not path.exists():
        return code_map
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            method_id = str(row["idx"])
            if method_id in wanted_ids:
                code_map[method_id] = row.get("func", "")
                if len(code_map) == len(wanted_ids):
                    break
    return code_map


def load_graph_manifest() -> dict:
    if not GRAPH_MANIFEST_PATH.exists():
        raise FileNotFoundError(f"Graph manifest not found: {GRAPH_MANIFEST_PATH}. Run ../run_pipeline/02_extract_graphs.py first.")
    return json.loads(GRAPH_MANIFEST_PATH.read_text(encoding="utf-8"))


def load_graphs_for_ids(manifest: dict, wanted_ids: set[str]) -> dict[str, dict[str, nx.DiGraph]]:
    graphs = {}
    for shard_path in manifest.get("shards", []):
        with Path(shard_path).open("rb") as f:
            shard = pickle.load(f)
        for method_id in list(wanted_ids - set(graphs)):
            if method_id in shard:
                graphs[method_id] = shard[method_id]
        if len(graphs) == len(wanted_ids):
            break
    return graphs


def load_spectral_features_for_ids(manifest_path: Path, wanted_ids: set[str]) -> dict[str, dict]:
    if not manifest_path.exists():
        return {}
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    features = {}
    for shard_path in manifest.get("shards", []):
        with Path(shard_path).open("rb") as f:
            shard = pickle.load(f)
        for method_id in list(wanted_ids - set(features)):
            if method_id in shard:
                features[method_id] = shard[method_id]
        if len(features) == len(wanted_ids):
            break
    return features


def draw_graph(graph: nx.DiGraph | None, title: str, ax, max_nodes: int = 100) -> None:
    ax.set_title(title, fontsize=9)
    ax.axis("off")
    if graph is None or graph.number_of_nodes() == 0:
        ax.text(0.5, 0.5, "empty", ha="center", va="center")
        return
    graph = nx.DiGraph(graph)
    shown = graph if graph.number_of_nodes() <= max_nodes else graph.subgraph(list(graph.nodes())[:max_nodes]).copy()
    pos = nx.spring_layout(shown, seed=42)
    nx.draw_networkx_edges(shown, pos, ax=ax, arrows=False, alpha=0.25, width=0.8)
    nx.draw_networkx_nodes(shown, pos, ax=ax, node_size=22, alpha=0.85)
    ax.text(0.01, 0.02, f"{graph.number_of_nodes()} nodes / {graph.number_of_edges()} edges", transform=ax.transAxes, fontsize=7)


def eigenvalues_for(spectral_map: dict, method_id: str, graph_type: str) -> np.ndarray:
    values = spectral_map.get(str(method_id), {}).get(graph_type, {}).get("eigenvalues", [])
    values = np.asarray(values, dtype=np.float64)
    return values[np.isfinite(values)]


def draw_eigenvalues(spectral_map: dict, method_id: str, graph_type: str, ax) -> None:
    values = eigenvalues_for(spectral_map, method_id, graph_type)
    ax.set_title(f"{graph_type.upper()} eigenvalues", fontsize=9)
    if values.size == 0:
        ax.text(0.5, 0.5, "missing", ha="center", va="center", transform=ax.transAxes)
        ax.set_xticks([])
        ax.set_yticks([])
        return
    ax.plot(np.arange(1, values.size + 1), np.sort(values), linewidth=1.2)
    ax.grid(alpha=0.2)
    ax.tick_params(labelsize=8)


def render_code(method_id: str, code_map: dict[str, str], title: str) -> None:
    display(HTML(f"""
    <div style="border:1px solid #d0d7de;border-radius:6px;overflow:hidden;margin:10px 0;">
      <div style="padding:6px 10px;background:#f6f8fa;font-family:system-ui,sans-serif;font-size:13px;">{html.escape(title)} - id {html.escape(str(method_id))}</div>
      <pre style="margin:0;padding:12px;overflow-x:auto;white-space:pre-wrap;font-size:12px;line-height:1.35;max-height:460px;">{html.escape(code_map.get(str(method_id), ""))}</pre>
    </div>
    """))


def render_code_pair(left_id: str, right_id: str, code_map: dict[str, str], title: str) -> None:
    display(HTML(f"""
    <h3>{html.escape(title)}</h3>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;align-items:start;">
      <div><b>left id: {html.escape(str(left_id))}</b><pre style="white-space:pre-wrap;font-size:12px;line-height:1.35;border:1px solid #d0d7de;padding:10px;border-radius:6px;max-height:460px;overflow:auto;">{html.escape(code_map.get(str(left_id), ""))}</pre></div>
      <div><b>right id: {html.escape(str(right_id))}</b><pre style="white-space:pre-wrap;font-size:12px;line-height:1.35;border:1px solid #d0d7de;padding:10px;border-radius:6px;max-height:460px;overflow:auto;">{html.escape(code_map.get(str(right_id), ""))}</pre></div>
    </div>
    """))


In [ ]:
def stage_runtime_row(stage: str) -> dict:
    timings = json.loads(PIPELINE_TIMINGS_PATH.read_text(encoding="utf-8")) if PIPELINE_TIMINGS_PATH.exists() else {"stages": {}}
    record = timings.get("stages", {}).get(stage, {})
    seconds = record.get("seconds")
    minutes = record.get("minutes")
    if minutes is None and isinstance(seconds, (int, float)):
        minutes = seconds / 60
    return {
        "stage": stage,
        "seconds": round(seconds, 2) if isinstance(seconds, (int, float)) else None,
        "minutes": round(minutes, 2) if isinstance(minutes, (int, float)) else None,
        "updated_at_utc": record.get("updated_at_utc"),
        "status": "recorded" if isinstance(seconds, (int, float)) else "not recorded yet",
        "source": str(PIPELINE_TIMINGS_PATH),
    }


In [ ]:
TUNING_JSON = OUTPUT_ROOT / f"trained_semantic_{LANGUAGE}_f1_pss_wasserstein.json"
PAIR_SCORES_CSV = OUTPUT_ROOT / f"pair_scores_trained_semantic_{LANGUAGE}_f1_pss_wasserstein.csv"
if not TUNING_JSON.exists():
    candidates = sorted(OUTPUT_ROOT.glob("trained_*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        TUNING_JSON = candidates[0]
if not PAIR_SCORES_CSV.exists():
    candidates = sorted(OUTPUT_ROOT.glob("pair_scores_trained_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        PAIR_SCORES_CSV = candidates[0]
for required in [TUNING_JSON, PAIR_SCORES_CSV, GRAPH_MANIFEST_PATH, SPECTRAL_MANIFEST_PATH]:
    if not required.exists():
        raise FileNotFoundError(f"Required artifact not found: {required}")

trained_df = pd.DataFrame(json.loads(TUNING_JSON.read_text(encoding="utf-8")))
if trained_df.empty:
    raise RuntimeError(f"No tuning rows found in {TUNING_JSON}")
if "decision_threshold" not in trained_df.columns and "best_threshold" in trained_df.columns:
    trained_df["decision_threshold"] = trained_df["best_threshold"].astype(float).map(lambda value: float(np.nextafter(value, -np.inf)))
sort_cols = [col for col in ["train_f1", "train_accuracy", "best_metric"] if col in trained_df.columns]
trained_df = trained_df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).reset_index(drop=True)
example_candidates = trained_df.copy()
if {"train_precision", "train_recall"}.issubset(example_candidates.columns):
    mixed = example_candidates[(example_candidates["train_precision"] < 1.0) & (example_candidates["train_recall"] < 1.0)]
    if not mixed.empty:
        example_candidates = mixed
best_config = example_candidates.iloc[0].to_dict()
ACTIVE_GRAPH = str(best_config["graph_type"])
ACTIVE_METRIC = str(best_config["metric"])
ACTIVE_K = "full" if pd.isna(best_config.get("k_eigen")) else str(best_config.get("k_eigen"))
ACTIVE_THRESHOLD = float(best_config["best_threshold"])
METRICS = sorted(trained_df["metric"].dropna().astype(str).unique())
available_graphs = set(trained_df["graph_type"].dropna().astype(str).unique())
GRAPH_TYPES = ['ast', 'cfg', 'ddg', 'pdg', 'cpg'] + sorted(available_graphs - set(GRAPH_TYPES))

print("Tuning JSON:", TUNING_JSON)
print("Pair scores CSV:", PAIR_SCORES_CSV)
print("Active config for TP/FP/FN/TN examples:", ACTIVE_GRAPH, ACTIVE_K, ACTIVE_METRIC, "threshold", ACTIVE_THRESHOLD)
display(trained_df)


In [ ]:
def decision_threshold(threshold: float) -> float:
    return float(np.nextafter(float(threshold), -np.inf))


def confusion_name(label: int, pred: int) -> str:
    if label == 1 and pred == 1:
        return "TP"
    if label == 0 and pred == 1:
        return "FP"
    if label == 1 and pred == 0:
        return "FN"
    return "TN"


def select_confusion_examples(csv_path: Path, graph_type: str, metric: str, k_label: str, threshold: float) -> dict[str, dict]:
    selected = {}
    usecols = ["left_id", "right_id", "label", "graph_type", "k_eigen", "metric", "score"]
    for chunk in pd.read_csv(csv_path, usecols=usecols, dtype={"left_id": str, "right_id": str, "k_eigen": str}, chunksize=250_000):
        chunk = chunk[(chunk["graph_type"].astype(str) == graph_type) & (chunk["metric"].astype(str) == metric) & (chunk["k_eigen"].astype(str) == k_label)]
        if chunk.empty:
            continue
        chunk = chunk.copy()
        chunk["pred"] = (chunk["score"].astype(float) >= decision_threshold(threshold)).astype(int)
        for row in chunk.itertuples(index=False):
            name = confusion_name(int(row.label), int(row.pred))
            if name not in selected:
                selected[name] = {"left_id": str(row.left_id), "right_id": str(row.right_id), "label": int(row.label), "pred": int(row.pred), "active_score": float(row.score)}
            if {"TP", "FP", "FN", "TN"}.issubset(selected):
                return selected
    return selected


examples = select_confusion_examples(PAIR_SCORES_CSV, ACTIVE_GRAPH, ACTIVE_METRIC, ACTIVE_K, ACTIVE_THRESHOLD)
print("Selected example types:", sorted(examples))
display(pd.DataFrame.from_dict(examples, orient="index"))
example_keys = {(item["left_id"], item["right_id"]) for item in examples.values()}
needed_ids = {method_id for key in example_keys for method_id in key}

score_rows = []
usecols = ["left_id", "right_id", "label", "graph_type", "k_eigen", "metric", "score"]
for chunk in pd.read_csv(PAIR_SCORES_CSV, usecols=usecols, dtype={"left_id": str, "right_id": str, "k_eigen": str}, chunksize=250_000):
    chunk = chunk[chunk["k_eigen"].astype(str) == ACTIVE_K]
    if chunk.empty:
        continue
    mask = [(left, right) in example_keys for left, right in zip(chunk["left_id"], chunk["right_id"])]
    if any(mask):
        score_rows.append(chunk.loc[mask].copy())
metric_scores_df = pd.concat(score_rows, ignore_index=True) if score_rows else pd.DataFrame(columns=usecols)
display(metric_scores_df.sort_values(["left_id", "right_id", "graph_type", "metric"]))


In [ ]:
code_map = load_code_for_ids(DATA_DIR / "data.jsonl", needed_ids)
graph_map = load_graphs_for_ids(load_graph_manifest(), needed_ids)
spectral_map = load_spectral_features_for_ids(SPECTRAL_MANIFEST_PATH, needed_ids)
print("Loaded code snippets:", len(code_map), "missing:", len(needed_ids - set(code_map)))
print("Loaded graph records:", len(graph_map), "missing:", len(needed_ids - set(graph_map)))
print("Loaded spectral records:", len(spectral_map), "missing:", len(needed_ids - set(spectral_map)))

for name, item in examples.items():
    title = f"{name}: label={item['label']} pred={item['pred']} selected by {ACTIVE_GRAPH}/{ACTIVE_METRIC} threshold={ACTIVE_THRESHOLD}"
    render_code_pair(item["left_id"], item["right_id"], code_map, title)
    display(metric_scores_df[(metric_scores_df["left_id"].astype(str) == item["left_id"]) & (metric_scores_df["right_id"].astype(str) == item["right_id"])].sort_values(["graph_type", "metric"]))
    fig, axes = plt.subplots(len(GRAPH_TYPES), 2, figsize=(10, max(8, 3.0 * len(GRAPH_TYPES))), squeeze=False)
    for row, graph_type in enumerate(GRAPH_TYPES):
        draw_graph(graph_map.get(item["left_id"], {}).get(graph_type), f"left {graph_type.upper()}", axes[row][0])
        draw_graph(graph_map.get(item["right_id"], {}).get(graph_type), f"right {graph_type.upper()}", axes[row][1])
    plt.tight_layout()
    plt.show()
    fig, axes = plt.subplots(len(GRAPH_TYPES), 1, figsize=(10, max(6, 2.1 * len(GRAPH_TYPES))), squeeze=False)
    for ax, graph_type in zip(axes[:, 0], GRAPH_TYPES):
        left_values = eigenvalues_for(spectral_map, item["left_id"], graph_type)
        right_values = eigenvalues_for(spectral_map, item["right_id"], graph_type)
        ax.set_title(f"{graph_type.upper()} eigenvalues", fontsize=9)
        if left_values.size:
            ax.plot(np.sort(left_values), label="left", linewidth=1.1)
        if right_values.size:
            ax.plot(np.sort(right_values), label="right", linewidth=1.1, alpha=0.85)
        if not left_values.size and not right_values.size:
            ax.text(0.5, 0.5, "missing", ha="center", va="center", transform=ax.transAxes)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


In [ ]:
def build_score_histograms(csv_path: Path, graph_types: list[str], metrics: list[str], k_label: str, bins: np.ndarray) -> dict:
    hist = {}
    usecols = ["graph_type", "k_eigen", "metric", "label", "score"]
    for chunk in pd.read_csv(csv_path, usecols=usecols, dtype={"k_eigen": str}, chunksize=500_000):
        chunk = chunk[chunk["k_eigen"].astype(str) == k_label]
        if chunk.empty:
            continue
        chunk["label"] = chunk["label"].astype(int)
        chunk["score"] = chunk["score"].astype(float).clip(0, 1)
        for graph in graph_types:
            graph_chunk = chunk[chunk["graph_type"].astype(str) == graph]
            for metric in metrics:
                metric_chunk = graph_chunk[graph_chunk["metric"].astype(str) == metric]
                for label in sorted({label for (_, _, label) in histograms.keys()}):
                    values = metric_chunk.loc[metric_chunk["label"] == label, "score"].to_numpy()
                    if values.size:
                        hist.setdefault((graph, metric, label), np.zeros(len(bins) - 1, dtype=np.int64))
                        hist[(graph, metric, label)] += np.histogram(values, bins=bins)[0]
    return hist


def plot_metric_distribution(metric: str, graph_types: list[str], histograms: dict, bins: np.ndarray) -> None:
    centers = (bins[:-1] + bins[1:]) / 2
    width = float(bins[1] - bins[0])
    fig, axes = plt.subplots(len(graph_types), 1, figsize=(11.5, max(3.0, 2.25 * len(graph_types))), sharex=True, squeeze=False)
    colors = {0: "#4C78A8", 1: "#F58518"}
    labels = {1: f"{DISPLAY_LANGUAGE} clone label 1"}
    for row, graph in enumerate(graph_types):
        ax = axes[row][0]
        max_y = 0.0
        for label in sorted({label for (_, _, label) in histograms.keys()}):
            counts = histograms.get((graph, metric, label), np.zeros(len(bins) - 1, dtype=np.int64))
            total = int(counts.sum())
            fraction = counts / total if total else counts.astype(float)
            max_y = max(max_y, float(fraction.max()) if fraction.size else 0.0)
            ax.bar(centers, fraction, width=width * 0.92, color=colors[label], alpha=0.58, edgecolor="white", linewidth=0.25, label=f"{labels.get(label, f'label {label}')} (n={total:,})")
        threshold_row = trained_df[(trained_df["graph_type"].astype(str) == graph) & (trained_df["metric"].astype(str) == metric) & (trained_df["k_eigen"].fillna("full").astype(str) == ACTIVE_K)]
        if not threshold_row.empty:
            threshold = float(threshold_row.iloc[0]["best_threshold"])
            ax.axvline(threshold, color="#222222", linestyle="--", linewidth=1.1, alpha=0.8)
            ax.text(threshold, max_y * 0.92 if max_y else 0.02, f"  th={threshold:.3f}", rotation=90, va="top", ha="left", fontsize=8, color="#222222")
        ax.set_title(f"{metric.upper()} distribution for {graph.upper()}", fontsize=11, pad=7)
        ax.set_ylabel("fraction of class", fontsize=9)
        ax.set_ylim(0, max(0.04, max_y * 1.18))
        ax.grid(True, axis="both", alpha=0.18, linewidth=0.8)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(loc="upper left", fontsize=8, frameon=True)
    axes[-1][0].set_xlabel(f"{metric} similarity", fontsize=10)
    axes[-1][0].set_xlim(0, 1)
    fig.suptitle(f"Semantic {DISPLAY_LANGUAGE} pair score distributions by label - {metric.upper()}", y=0.995, fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.985])
    plt.show()


bins = np.linspace(0.0, 1.0, 71)
histograms = build_score_histograms(PAIR_SCORES_CSV, GRAPH_TYPES, METRICS, ACTIVE_K, bins)
for metric in METRICS:
    plot_metric_distribution(metric, GRAPH_TYPES, histograms, bins)


## Runtime Summary

In [ ]:
time_cols = ["graph_type", "metric", "k_eigen", "score_time_seconds", "threshold_time_seconds", "total_config_time_seconds"]
missing = [col for col in time_cols if col not in trained_df.columns]
if missing:
    display(HTML("<b>Timing per graph/metric is not available in this tuning JSON.</b> Re-run <code>04_tune_pss_wasserstein.py</code> to refresh timing fields."))
else:
    timing_df = trained_df[time_cols].copy()
    timing_df["k_eigen"] = timing_df["k_eigen"].where(timing_df["k_eigen"].notna(), "full")
    display(timing_df.sort_values(["metric", "graph_type"]))
    display(timing_df.groupby("metric", as_index=False)[["score_time_seconds", "threshold_time_seconds", "total_config_time_seconds"]].sum().sort_values("total_config_time_seconds", ascending=False))

display(pd.DataFrame([stage_runtime_row("04_tune_pss_wasserstein")]))
